# SurajClaw Agent Lab

Django-free reference rebuild of the entire SurajClaw multi-agent stack.

This notebook mirrors `surajclaw/agents/` cell-for-cell so we can:

1. Reproduce the "what are my emails" misrouting bug.
2. Iterate on the General Agent system prompt + delegation tool docstrings until Gemini routes correctly.
3. Port the validated prompt + docstring changes back into `surajclaw/agents/registry.py` and `surajclaw/agents/subgraphs/general.py` mechanically.

No `django.setup()` anywhere. Real Google APIs via `google_tokens/<label>.json` written by the production OAuth flow. Real Gemini via `langchain-google-genai`. Memory tool is an in-notebook stub (toy list + Gemini embeddings) since the routing question doesn't need pgvector.

## 1. Setup

Load `.env` (lives at the SurajClaw repo root, one level above `surajclaw/`), assert credentials, import the LangGraph + LangChain + Google client surfaces. Picks up `GOOGLE_ACCOUNT_LABEL` (defaults to `suraj2308`, the account already connected per the OAuth callback log).

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
SURAJCLAW_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REPO_ROOT = SURAJCLAW_DIR.parent
ENV_PATH = REPO_ROOT / ".env"
TOKEN_DIR = SURAJCLAW_DIR / "google_tokens"

load_dotenv(ENV_PATH)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.0-flash")
GOOGLE_ACCOUNT_LABEL = os.environ.get("GOOGLE_ACCOUNT_LABEL", "suraj2308")

assert GEMINI_API_KEY, f"GEMINI_API_KEY missing in {ENV_PATH}"
assert TOKEN_DIR.exists(), f"google_tokens dir missing: {TOKEN_DIR}"

print(f"env:    {ENV_PATH}")
print(f"tokens: {TOKEN_DIR} -> {sorted(p.name for p in TOKEN_DIR.glob('*.json'))}")
print(f"model:  {GEMINI_MODEL}")
print(f"label:  {GOOGLE_ACCOUNT_LABEL}")

env:    /Users/sudarshanrajagopalan/Developer/SurajClaw/.env
tokens: /Users/sudarshanrajagopalan/Developer/SurajClaw/surajclaw/google_tokens -> ['suraj2308.json']
model:  gemini-3.1-flash-lite-preview
label:  suraj2308


## 2. Types

Mirrors `surajclaw/agents/types.py` exactly so the notebook objects are drop-in compatible with the production code.

In [2]:
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any, Literal

AgentStatus = Literal["ok", "needs_approval", "failed", "handoff", "retry"]
ToolRiskLevel = Literal["low", "medium", "high"]


@dataclass(frozen=True)
class AgentDefinition:
    id: str
    display_name: str
    description: str
    graph_factory: Callable[[], Any]
    system_prompt: str
    default_model_provider: str = "gemini"
    allowed_tools: frozenset[str] = field(default_factory=frozenset)
    direct_access: bool = True
    delegatable: bool = True
    max_steps: int = 6


@dataclass
class AgentInvocation:
    session_id: str
    source: str
    agent_id: str
    task: str
    account_label: str | None = None
    context: dict[str, Any] = field(default_factory=dict)


@dataclass
class AgentResult:
    agent_id: str
    status: AgentStatus
    output: str
    structured: dict[str, Any] = field(default_factory=dict)
    next_agent: str | None = None
    approval_request_id: str | None = None


@dataclass(frozen=True)
class ToolDefinition:
    id: str
    callable: Callable[..., dict[str, Any]]
    description: str
    parameters: dict[str, Any] = field(default_factory=dict)
    risk_level: ToolRiskLevel = "low"
    approval_required: bool = False
    required_env: tuple[str, ...] = ()


@dataclass
class ToolResult:
    ok: bool
    output: str
    structured: dict[str, Any] = field(default_factory=dict)
    error: str | None = None
    approved_by: str | None = None
    was_gated: bool = False

## 3. Google credentials helper

Same shape as `surajclaw/core/google_accounts.py:GoogleAccount.load_credentials` but reads directly from disk so we can skip Django settings.

In [3]:
import json

from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build


def creds_for(label: str) -> Credentials:
    """Read google_tokens/<label>.json and return refreshed Credentials."""
    token_path = TOKEN_DIR / f"{label}.json"
    if not token_path.exists():
        raise LookupError(
            f"no token for account {label!r} at {token_path}. "
            f"Run: python manage.py google_oauth_login --account {label}"
        )
    data = json.loads(token_path.read_text(encoding="utf-8"))
    return Credentials.from_authorized_user_info(data)


def google_service(label: str, name: str, version: str):
    return build(name, version, credentials=creds_for(label), cache_discovery=False)


print("creds_for OK -> account:", GOOGLE_ACCOUNT_LABEL)
_creds_smoke = creds_for(GOOGLE_ACCOUNT_LABEL)
print("scopes:", _creds_smoke.scopes)

creds_for OK -> account: suraj2308
scopes: ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/calendar', 'https://www.googleapis.com/auth/tasks', 'https://www.googleapis.com/auth/drive.file', 'https://www.googleapis.com/auth/documents', 'https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/contacts.readonly', 'openid', 'https://www.googleapis.com/auth/userinfo.email']


## 4a. Google Workspace tool callables

Direct port of `surajclaw/tools/google/workspace.py` minus the Django-bound `_service` helper. Same return shape (`{"ok": bool, "output": str, "structured": dict}`).

In [4]:
def list_accounts() -> dict:
    labels = sorted(p.stem for p in TOKEN_DIR.glob("*.json"))
    return {"ok": True, "output": "accounts: " + ", ".join(labels), "structured": {"accounts": labels}}


def gmail_search_messages(account_label: str, query: str = "", limit: int = 10) -> dict:
    service = google_service(account_label, "gmail", "v1")
    response = service.users().messages().list(userId="me", q=query, maxResults=limit).execute()
    messages = response.get("messages", [])
    return {"ok": True, "output": f"Found {len(messages)} Gmail messages.", "structured": {"messages": messages}}


def gmail_get_message(account_label: str, message_id: str) -> dict:
    service = google_service(account_label, "gmail", "v1")
    message = service.users().messages().get(userId="me", id=message_id, format="full").execute()
    return {"ok": True, "output": message.get("snippet", ""), "structured": {"message": message}}


def gmail_list_threads(account_label: str, query: str = "", limit: int = 10) -> dict:
    service = google_service(account_label, "gmail", "v1")
    response = service.users().threads().list(userId="me", q=query, maxResults=limit).execute()
    threads = response.get("threads", [])
    return {"ok": True, "output": f"Found {len(threads)} Gmail threads.", "structured": {"threads": threads}}


def calendar_list_events(account_label: str, query: str = "", limit: int = 10) -> dict:
    service = google_service(account_label, "calendar", "v3")
    events = (
        service.events()
        .list(calendarId="primary", q=query or None, maxResults=limit, singleEvents=True)
        .execute()
        .get("items", [])
    )
    return {"ok": True, "output": f"Found {len(events)} calendar events.", "structured": {"events": events}}


def calendar_create_event(account_label: str, summary: str, start: dict | None = None, end: dict | None = None) -> dict:
    service = google_service(account_label, "calendar", "v3")
    body: dict[str, Any] = {"summary": summary}
    if start and end:
        body.update({"start": start, "end": end})
    event = service.events().insert(calendarId="primary", body=body).execute()
    return {"ok": True, "output": f"Created calendar event {event.get('id')}", "structured": {"event": event}}


def calendar_update_event(account_label: str, event_id: str, summary: str | None = None, patch: dict | None = None) -> dict:
    service = google_service(account_label, "calendar", "v3")
    body = patch or {}
    if summary:
        body["summary"] = summary
    event = service.events().patch(calendarId="primary", eventId=event_id, body=body).execute()
    return {"ok": True, "output": f"Updated calendar event {event.get('id')}", "structured": {"event": event}}


def calendar_delete_event(account_label: str, event_id: str) -> dict:
    google_service(account_label, "calendar", "v3").events().delete(calendarId="primary", eventId=event_id).execute()
    return {"ok": True, "output": f"Deleted calendar event {event_id}"}


def tasks_list_tasklists(account_label: str) -> dict:
    lists = google_service(account_label, "tasks", "v1").tasklists().list().execute().get("items", [])
    return {"ok": True, "output": f"Found {len(lists)} task lists.", "structured": {"tasklists": lists}}


def tasks_list_tasks(account_label: str, tasklist_id: str = "@default", limit: int = 10) -> dict:
    tasks = (
        google_service(account_label, "tasks", "v1")
        .tasks()
        .list(tasklist=tasklist_id, maxResults=limit)
        .execute()
        .get("items", [])
    )
    return {"ok": True, "output": f"Found {len(tasks)} tasks.", "structured": {"tasks": tasks}}


def tasks_create_task(account_label: str, title: str, tasklist_id: str = "@default") -> dict:
    task = (
        google_service(account_label, "tasks", "v1")
        .tasks()
        .insert(tasklist=tasklist_id, body={"title": title})
        .execute()
    )
    return {"ok": True, "output": f"Created task {task.get('id')}", "structured": {"task": task}}


def drive_search_files(account_label: str, query: str = "", limit: int = 10) -> dict:
    q = query or "trashed = false"
    files = (
        google_service(account_label, "drive", "v3")
        .files()
        .list(q=q, pageSize=limit, fields="files(id,name,mimeType,webViewLink)")
        .execute()
        .get("files", [])
    )
    return {"ok": True, "output": f"Found {len(files)} Drive files.", "structured": {"files": files}}


def docs_create_doc(account_label: str, title: str) -> dict:
    doc = google_service(account_label, "docs", "v1").documents().create(body={"title": title}).execute()
    return {"ok": True, "output": f"Created doc {doc.get('documentId')}", "structured": {"document": doc}}


def sheets_create_sheet(account_label: str, title: str) -> dict:
    sheet = (
        google_service(account_label, "sheets", "v4")
        .spreadsheets()
        .create(body={"properties": {"title": title}})
        .execute()
    )
    return {"ok": True, "output": f"Created sheet {sheet.get('spreadsheetId')}", "structured": {"spreadsheet": sheet}}


def contacts_search(account_label: str, query: str = "", limit: int = 10) -> dict:
    response = (
        google_service(account_label, "people", "v1")
        .people()
        .searchContacts(query=query, readMask="names,emailAddresses", pageSize=limit)
        .execute()
    )
    results = response.get("results", [])
    return {"ok": True, "output": f"Found {len(results)} contacts.", "structured": {"results": results}}


print("Google tool callables defined.")

Google tool callables defined.


## 4b. Other tool callables (web, memory, notes, sandbox, workspace)

Memory is the most interesting one for the routing bug: it uses a tiny in-notebook list of "remembered facts" + Gemini embedding cosine similarity, so we can prove the model is picking it inappropriately rather than blame production retrieval.

In [5]:
import math
import shlex
import subprocess

import httpx

GOOGLE_SEARCH_API_KEY = os.environ.get("GOOGLE_SEARCH_API_KEY", "")
GOOGLE_SEARCH_CX = os.environ.get("GOOGLE_SEARCH_CX", "")

WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

NOTES_DIR = SURAJCLAW_DIR / "notes"
NOTES_DIR.mkdir(parents=True, exist_ok=True)

MEMORY_STORE: list[dict[str, Any]] = [
    {"text": "Project status: Q3 OKRs frozen, demo on May 12.", "source": "saved_note"},
    {"text": "Suraj prefers concise summaries with bullet points.", "source": "saved_note"},
]


def _embed(text: str) -> list[float]:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embedder = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GEMINI_API_KEY)
    return embedder.embed_query(text)


def _cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0


def web_search(query: str, limit: int = 5) -> dict:
    if not GOOGLE_SEARCH_API_KEY or not GOOGLE_SEARCH_CX:
        return {"ok": False, "output": "GOOGLE_SEARCH_API_KEY and GOOGLE_SEARCH_CX are required.", "error": "missing_env"}
    response = httpx.get(
        "https://www.googleapis.com/customsearch/v1",
        params={"key": GOOGLE_SEARCH_API_KEY, "cx": GOOGLE_SEARCH_CX, "q": query, "num": max(1, min(limit, 10))},
        timeout=15.0,
    )
    response.raise_for_status()
    items = response.json().get("items", [])
    lines = [f"{i.get('title', '?')} - {i.get('link', '')}\n{i.get('snippet', '')}" for i in items]
    return {"ok": True, "output": "\n\n".join(lines) or "No results.", "structured": {"items": items}}


def memory_search(query: str, limit: int = 5) -> dict:
    if not MEMORY_STORE:
        return {"ok": True, "output": "No relevant memory found.", "structured": {"hits": []}}
    qvec = _embed(query)
    scored = [(_cosine(qvec, _embed(item["text"])), item) for item in MEMORY_STORE]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[: max(1, min(limit, 10))]
    if not top or top[0][0] < 0.55:
        return {"ok": True, "output": "No relevant memory found.", "structured": {"hits": []}}
    lines = [f"[{score:.2f}] {item['text']}" for score, item in top]
    return {"ok": True, "output": "\n".join(lines), "structured": {"hits": [item for _, item in top]}}


def notes_write(title: str, content: str, session_id: str | None = None) -> dict:
    slug = "-".join(title.lower().split())[:60] or "note"
    path = NOTES_DIR / f"{slug}.md"
    path.write_text(content, encoding="utf-8")
    return {"ok": True, "output": f"Wrote note {title} ({path.name}).", "structured": {"path": str(path)}}


def notes_search(query: str, limit: int = 5) -> dict:
    notes = sorted(NOTES_DIR.glob("*.md"))
    hits = [n for n in notes if query.lower() in n.read_text(encoding="utf-8").lower()][:limit]
    lines = [f"{n.stem}\n{n.read_text(encoding='utf-8')[:200]}" for n in hits]
    return {"ok": True, "output": "\n\n".join(lines) or "No notes found.", "structured": {"notes": [n.name for n in hits]}}


def notes_list(limit: int = 10) -> dict:
    notes = sorted(NOTES_DIR.glob("*.md"))[:limit]
    return {"ok": True, "output": "\n".join(n.stem for n in notes) or "No notes.", "structured": {"notes": [n.name for n in notes]}}


def workspace_read_file(path: str, max_chars: int = 12000) -> dict:
    target = (WORKSPACE_DIR / path).resolve()
    if WORKSPACE_DIR.resolve() not in target.parents and target != WORKSPACE_DIR.resolve():
        return {"ok": False, "output": "path escapes WORKSPACE_DIR", "error": "bad_path"}
    if not target.is_file():
        return {"ok": False, "output": f"file not found: {path}", "error": "not_found"}
    text = target.read_text(encoding="utf-8", errors="replace")[:max_chars]
    return {"ok": True, "output": text, "structured": {"path": str(target)}}


def workspace_write_file(path: str, content: str) -> dict:
    target = (WORKSPACE_DIR / path).resolve()
    if WORKSPACE_DIR.resolve() not in target.parents and target != WORKSPACE_DIR.resolve():
        return {"ok": False, "output": "path escapes WORKSPACE_DIR", "error": "bad_path"}
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return {"ok": True, "output": f"wrote {path}", "structured": {"path": str(target)}}


def workspace_list_files(path: str = ".", limit: int = 50) -> dict:
    target = (WORKSPACE_DIR / path).resolve()
    if not target.exists():
        return {"ok": False, "output": f"path not found: {path}", "error": "not_found"}
    items = sorted(target.iterdir(), key=lambda p: p.name)[:limit] if target.is_dir() else [target]
    return {"ok": True, "output": "\n".join(p.name for p in items), "structured": {"files": [p.name for p in items]}}


def sandbox_run_shell(command: str, timeout_seconds: int | None = 30, session_id: str | None = None) -> dict:
    proc = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout_seconds)
    return {
        "ok": proc.returncode == 0,
        "output": proc.stdout or proc.stderr,
        "structured": {"stdout": proc.stdout, "stderr": proc.stderr, "code": proc.returncode},
        "error": None if proc.returncode == 0 else "nonzero_exit",
    }


def sandbox_run_python(command: str, timeout_seconds: int | None = 30, session_id: str | None = None) -> dict:
    proc = subprocess.run(["python3", "-c", command], capture_output=True, text=True, timeout=timeout_seconds)
    return {
        "ok": proc.returncode == 0,
        "output": proc.stdout or proc.stderr,
        "structured": {"stdout": proc.stdout, "stderr": proc.stderr, "code": proc.returncode},
        "error": None if proc.returncode == 0 else "nonzero_exit",
    }


def sandbox_read_file(path: str, session_id: str | None = None) -> dict:
    return sandbox_run_shell(f"sed -n '1,200p' {shlex.quote(path)}", session_id=session_id)


def sandbox_write_file(path: str, content: str, session_id: str | None = None) -> dict:
    return sandbox_run_shell(f"printf %s {shlex.quote(content)} > {shlex.quote(path)}", session_id=session_id)


def sandbox_run_tests(command: str = "pytest", timeout_seconds: int | None = 60, session_id: str | None = None) -> dict:
    return sandbox_run_shell(command, timeout_seconds=timeout_seconds, session_id=session_id)


print("Other tool callables defined.")

Other tool callables defined.


## 5. Tool registry

Notebook-local mirror of `surajclaw/tools/registry.py` minus the audit/approval Django bits. `get_langchain_tools(agent_id)` returns one `StructuredTool` per allowed tool, with the same Pydantic-derived args schema the production code uses.

In [6]:
import inspect
import re
from typing import get_type_hints

from langchain_core.tools import StructuredTool
from pydantic import Field, create_model

_TOOLS: dict[str, ToolDefinition] = {}


def register_tool(tool: ToolDefinition) -> ToolDefinition:
    _TOOLS[tool.id] = tool
    return tool


def get_tool(tool_id: str) -> ToolDefinition:
    return _TOOLS[tool_id]


def list_tools() -> list[ToolDefinition]:
    return sorted(_TOOLS.values(), key=lambda t: t.id)


def list_tools_for_agent(agent_id: str) -> list[ToolDefinition]:
    agent = get_agent(agent_id)
    return [t for t in list_tools() if t.id in agent.allowed_tools]


def execute_tool(*, agent_id: str, tool_id: str, args: dict[str, Any], session_id: str) -> ToolResult:
    agent = get_agent(agent_id)
    if tool_id not in agent.allowed_tools:
        return ToolResult(ok=False, output=f"agent `{agent_id}` cannot use `{tool_id}`", error="tool_not_allowed")
    tool = get_tool(tool_id)
    call_args = dict(args)
    if "session_id" in inspect.signature(tool.callable).parameters:
        call_args.setdefault("session_id", session_id)
    try:
        raw = tool.callable(**call_args)
    except Exception as exc:
        return ToolResult(ok=False, output=f"tool failed: {exc}", error=type(exc).__name__)
    return ToolResult(
        ok=bool(raw.get("ok", True)),
        output=str(raw.get("output", "")),
        structured=raw.get("structured", {}) or {},
        error=raw.get("error"),
    )


def _safe_tool_name(tool_id: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9_]", "_", tool_id)
    if not name or name[0].isdigit():
        name = f"tool_{name}"
    return name[:64]


def get_langchain_tools(agent_id: str, session_id: str | None = None) -> list[StructuredTool]:
    tools: list[StructuredTool] = []
    for tool in list_tools_for_agent(agent_id):
        fields: dict[str, tuple[Any, Any]] = {}
        type_hints = get_type_hints(tool.callable)
        for name, param in inspect.signature(tool.callable).parameters.items():
            if name == "session_id":
                continue
            annotation = type_hints.get(name, Any)
            default = param.default if param.default is not inspect._empty else ...
            fields[name] = (annotation, Field(default=default))
        args_schema = create_model(f"{_safe_tool_name(tool.id)}Args", **fields)

        def _runner(_tool_id=tool.id, **kwargs):
            result = execute_tool(agent_id=agent_id, tool_id=_tool_id, args=kwargs, session_id=session_id or "")
            return result.output if result.ok else f"ERROR ({result.error or 'tool_failed'}): {result.output}"

        tools.append(
            StructuredTool.from_function(
                func=_runner,
                name=_safe_tool_name(tool.id),
                description=f"{tool.description}\nOriginal tool id: {tool.id}",
                args_schema=args_schema,
            )
        )
    return tools


for _id, _fn, _desc in (
    ("google.accounts.list", list_accounts, "List connected Google accounts."),
    ("google.gmail.search_messages", gmail_search_messages, "Search Gmail messages read-only."),
    ("google.gmail.get_message", gmail_get_message, "Read one Gmail message."),
    ("google.gmail.list_threads", gmail_list_threads, "List Gmail threads read-only."),
    ("google.calendar.list_events", calendar_list_events, "List calendar events."),
    ("google.calendar.create_event", calendar_create_event, "Create calendar event."),
    ("google.calendar.update_event", calendar_update_event, "Update calendar event."),
    ("google.calendar.delete_event", calendar_delete_event, "Delete calendar event."),
    ("google.tasks.list_tasklists", tasks_list_tasklists, "List task lists."),
    ("google.tasks.list_tasks", tasks_list_tasks, "List tasks."),
    ("google.tasks.create_task", tasks_create_task, "Create task."),
    ("google.drive.search_files", drive_search_files, "Search Drive files."),
    ("google.docs.create_doc", docs_create_doc, "Create Google Doc."),
    ("google.sheets.create_sheet", sheets_create_sheet, "Create Google Sheet."),
    ("google.contacts.search", contacts_search, "Search contacts read-only."),
    ("web.search", web_search, "Search the web with Google Custom Search."),
    ("memory.search", memory_search, "Search notes, session summaries, and entities."),
    ("notes.write", notes_write, "Write or update a Markdown note and index it."),
    ("notes.search", notes_search, "Search indexed Markdown notes semantically."),
    ("notes.list", notes_list, "List recent notes."),
    ("workspace.read_file", workspace_read_file, "Read a file under WORKSPACE_DIR."),
    ("workspace.write_file", workspace_write_file, "Write a file under WORKSPACE_DIR."),
    ("workspace.list_files", workspace_list_files, "List files under WORKSPACE_DIR."),
    ("sandbox.run_shell", sandbox_run_shell, "Run a shell command in sandbox."),
    ("sandbox.run_python", sandbox_run_python, "Run Python in sandbox."),
    ("sandbox.read_file", sandbox_read_file, "Read a file in sandbox."),
    ("sandbox.write_file", sandbox_write_file, "Write a file in sandbox."),
    ("sandbox.run_tests", sandbox_run_tests, "Run tests in sandbox."),
):
    register_tool(ToolDefinition(_id, _fn, _desc))

print(f"Registered {len(_TOOLS)} tools.")

Registered 28 tools.


## 6. Agent registry

Same `AgentDefinition` rows as `surajclaw/agents/registry.py`, but with the **real** prompt text on `system_prompt` (not the dead one-liners). The subgraphs in cell 8 read from this registry instead of using module-level constants — that's the de-duplication we'll port back in Phase 2.

These initial prompts are deliberately the **buggy** ones (verbatim from the production subgraph files). Cell 11 will rewrite them.

In [7]:
GENERAL_PROMPT_V1 = """You are SurajClaw's General Agent.

You are the default brain for a single-user personal AI system. Answer directly
when no tool is needed. Use direct tools for web search, memory lookup,
workspace files, and sandboxed code execution. Delegate specialized work to the
Google Workspace, Code Executor, or Notes agents when they are better suited.
Do not claim an external action succeeded unless a tool or subagent confirms it.
"""

GOOGLE_PROMPT_V1 = """You are SurajClaw's Google Workspace specialist.

Use Google Workspace tools for Gmail, Calendar, Tasks, Drive, Docs, Sheets, and
Contacts. Prefer read/search before write/update/delete. Gmail is read-only in
this system. Use the supplied account_label when a tool requires an account.
"""

CODE_PROMPT_V1 = """You are SurajClaw's Code Executor specialist.

Use only sandbox tools for shell commands, Python, file reads/writes, and tests.
Extract the command or code from the user's natural language request before
running it. Summarize stdout, stderr, exit codes, and any next debugging step.
"""

NOTES_PROMPT_V1 = """You are SurajClaw's Notes Agent.

Search existing notes before creating duplicates. For research tasks, search the
web when needed, synthesize the useful facts, and write clear Markdown notes.
Keep notes concise, source-aware, and easy to retrieve later.
"""

_AGENTS: dict[str, AgentDefinition] = {}


def register_agent(definition: AgentDefinition) -> None:
    _AGENTS[definition.id] = definition


def get_agent(agent_id: str) -> AgentDefinition:
    return _AGENTS[agent_id]


def list_agents() -> list[AgentDefinition]:
    return sorted(_AGENTS.values(), key=lambda a: a.id)


def reset_agents(general_prompt: str = GENERAL_PROMPT_V1,
                 google_prompt: str = GOOGLE_PROMPT_V1,
                 code_prompt: str = CODE_PROMPT_V1,
                 notes_prompt: str = NOTES_PROMPT_V1) -> None:
    """Re-register all agents (lets us hot-swap prompts in cell 11)."""
    _AGENTS.clear()
    register_agent(AgentDefinition(
        id="general",
        display_name="General Agent",
        description="Default supervisor.",
        graph_factory=lambda: build_general_graph(),
        system_prompt=general_prompt,
        allowed_tools=frozenset({
            "web.search", "memory.search",
            "workspace.read_file", "workspace.write_file", "workspace.list_files",
            "sandbox.run_shell", "sandbox.run_python", "sandbox.read_file",
            "sandbox.write_file", "sandbox.run_tests",
        }),
        max_steps=12,
        delegatable=False,
    ))
    register_agent(AgentDefinition(
        id="google_workspace",
        display_name="Google Workspace Agent",
        description="Gmail / Calendar / Drive / Docs / Sheets / Tasks / Contacts.",
        graph_factory=lambda: build_google_workspace_graph(),
        system_prompt=google_prompt,
        allowed_tools=frozenset({
            "google.accounts.list",
            "google.gmail.search_messages", "google.gmail.get_message", "google.gmail.list_threads",
            "google.calendar.list_events", "google.calendar.create_event",
            "google.calendar.update_event", "google.calendar.delete_event",
            "google.tasks.list_tasklists", "google.tasks.list_tasks", "google.tasks.create_task",
            "google.drive.search_files", "google.docs.create_doc",
            "google.sheets.create_sheet", "google.contacts.search",
        }),
    ))
    register_agent(AgentDefinition(
        id="code_executor",
        display_name="Code Executor Sandbox Agent",
        description="Sandbox shell/python/test runner.",
        graph_factory=lambda: build_code_executor_graph(),
        system_prompt=code_prompt,
        allowed_tools=frozenset({
            "sandbox.run_shell", "sandbox.run_python", "sandbox.read_file",
            "sandbox.write_file", "sandbox.run_tests",
        }),
    ))
    register_agent(AgentDefinition(
        id="notes",
        display_name="Notes Agent",
        description="Markdown note research and writing.",
        graph_factory=lambda: build_notes_graph(),
        system_prompt=notes_prompt,
        allowed_tools=frozenset({
            "notes.write", "notes.search", "notes.list",
            "web.search", "workspace.write_file", "memory.search",
        }),
    ))


print("Agent registry helpers defined.")

Agent registry helpers defined.


## 7. ReAct runner

Port of `surajclaw/agents/subgraphs/reactive.py`. Same `_compose_prompt`, same streaming via `stream_mode=["messages", "updates"]`, same tool-call frame extraction. `format_context` is dropped (no Django memory service); we just pass the raw system prompt through.

In [8]:
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent


class AgentState(TypedDict, total=False):
    session_id: str
    source: str
    user_message: str
    tool_calls: list[dict[str, Any]]
    tool_results: list[dict[str, Any]]
    context: dict[str, Any]
    messages: list[dict[str, Any]]
    agent_messages: Annotated[list[BaseMessage], add_messages]
    final_response: str
    active_agent: str
    requested_agent: str | None
    agent_trace: list[dict[str, Any]]
    agent_results: list[dict[str, Any]]
    last_agent_result: dict[str, Any]
    done: bool
    step_count: int
    max_steps: int
    account_label: str | None
    agent_result: dict[str, Any]
    model_used: str | None


def build_llm() -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        api_key=GEMINI_API_KEY,
        temperature=0.2,
        convert_system_message_to_human=True,
    )


def _compose_prompt(system_prompt: str) -> str:
    prompt = system_prompt.strip()
    prompt += (
        "\n\nUse tools only when they materially improve the answer. "
        "If a tool returns an error, explain the limitation and choose the next safest step."
    )
    return prompt


def _message_content(message: Any) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or item))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def _safe_args(args: Any) -> Any:
    if args is None:
        return {}
    if isinstance(args, (str, int, float, bool, list, dict)):
        try:
            json.dumps(args)
            return args
        except TypeError:
            return str(args)
    return str(args)


def _stream_react_agent(agent, payload, config, on_event) -> list[Any]:
    final_messages: list[Any] = []
    streamed_call_ids: set[str] = set()
    events = agent.stream(payload, config, stream_mode=["messages", "updates"])
    for mode, chunk in events:
        if mode == "messages":
            try:
                message_chunk, metadata = chunk
            except (TypeError, ValueError):
                continue
            node = (metadata or {}).get("langgraph_node")
            if node and node != "agent":
                continue
            text = _message_content(message_chunk)
            if text:
                on_event({"type": "token", "content": text})
        elif mode == "updates":
            for node_name, node_state in (chunk or {}).items():
                if not isinstance(node_state, dict):
                    continue
                for message in node_state.get("messages") or []:
                    if node_name == "agent":
                        for call in getattr(message, "tool_calls", None) or []:
                            cid = call.get("id") or ""
                            if cid in streamed_call_ids:
                                continue
                            if cid:
                                streamed_call_ids.add(cid)
                            on_event({
                                "type": "tool_call",
                                "name": call.get("name", ""),
                                "args": _safe_args(call.get("args")),
                                "id": cid,
                            })
                        final_messages.append(message)
                    elif node_name == "tools":
                        on_event({
                            "type": "tool_result",
                            "name": getattr(message, "name", "") or "",
                            "id": getattr(message, "tool_call_id", "") or "",
                            "content": _message_content(message)[:1200],
                        })
                        final_messages.append(message)
    return final_messages


def run_react_agent(*, state: AgentState, agent_id: str, system_prompt: str, tools: list[Any]) -> AgentState:
    task = state.get("user_message", "")
    context = state.get("context") or {}
    on_event = context.get("on_event")
    llm = build_llm()
    prompt = _compose_prompt(system_prompt)
    agent = create_react_agent(llm, tools, prompt=prompt)
    config = {"recursion_limit": max(8, state.get("max_steps", 6) * 4)}
    payload = {"messages": [HumanMessage(content=task)]}
    if on_event is None:
        result = agent.invoke(payload, config)
        messages = result.get("messages", [])
    else:
        messages = _stream_react_agent(agent, payload, config, on_event)
    output = _message_content(messages[-1]) if messages else ""
    state["agent_messages"] = messages
    state["final_response"] = output
    state["model_used"] = GEMINI_MODEL
    state["agent_result"] = {
        "status": "ok",
        "output": output,
        "structured": {"model_used": GEMINI_MODEL},
    }
    return state


print("ReAct runner defined.")

ReAct runner defined.


## 8. Subgraphs

One factory per agent — `general` plus the three specialists. Each reads `system_prompt` from the registry instead of a module-level constant. The general subgraph keeps the `_subagent_tools` factory that exposes `invoke_google_workspace`, `invoke_code_executor`, `invoke_notes_agent` as `StructuredTool`s; the docstrings here are the **buggy** thin originals so we can reproduce the misrouting.

In [9]:
DELEGATION_DOCS_V1 = {
    "google_workspace": "Delegate Google Workspace tasks like Gmail, Calendar, Drive, Docs, Sheets, Tasks, and Contacts.",
    "code_executor": "Delegate code execution, shell commands, debugging, or test runs to the sandbox specialist.",
    "notes": "Delegate note writing, note search, or research-to-note tasks.",
}

CURRENT_DELEGATION_DOCS: dict[str, str] = dict(DELEGATION_DOCS_V1)


def _invoke_subagent(*, session_id: str, source: str, agent_id: str, task: str, context: dict, account_label: str | None) -> str:
    result = invoke_agent(AgentInvocation(
        session_id=session_id,
        source=source,
        agent_id=agent_id,
        task=task,
        account_label=account_label,
        context={**context, "delegated_by": "general"},
    ))
    return f"{agent_id} [{result.status}]: {result.output}"


def _subagent_tools(state: AgentState) -> list[StructuredTool]:
    session_id = state["session_id"]
    source = state.get("source", "web")
    context = dict(state.get("context") or {})
    account_label = state.get("account_label") or context.get("account_label") or GOOGLE_ACCOUNT_LABEL

    def invoke_google_workspace(task: str, account_label: str = account_label) -> str:
        return _invoke_subagent(
            session_id=session_id, source=source, agent_id="google_workspace",
            task=task, context=context, account_label=account_label or "default",
        )

    def invoke_code_executor(task: str) -> str:
        return _invoke_subagent(
            session_id=session_id, source=source, agent_id="code_executor",
            task=task, context=context, account_label=account_label,
        )

    def invoke_notes_agent(task: str) -> str:
        return _invoke_subagent(
            session_id=session_id, source=source, agent_id="notes",
            task=task, context=context, account_label=account_label,
        )

    invoke_google_workspace.__doc__ = CURRENT_DELEGATION_DOCS["google_workspace"]
    invoke_code_executor.__doc__ = CURRENT_DELEGATION_DOCS["code_executor"]
    invoke_notes_agent.__doc__ = CURRENT_DELEGATION_DOCS["notes"]

    return [
        StructuredTool.from_function(invoke_google_workspace),
        StructuredTool.from_function(invoke_code_executor),
        StructuredTool.from_function(invoke_notes_agent),
    ]


def _general_node(state: AgentState) -> AgentState:
    session_id = state["session_id"]
    tools = get_langchain_tools("general", session_id=session_id)
    tools.extend(_subagent_tools(state))
    return run_react_agent(
        state=state, agent_id="general",
        system_prompt=get_agent("general").system_prompt, tools=tools,
    )


def _google_workspace_node(state: AgentState) -> AgentState:
    return run_react_agent(
        state=state, agent_id="google_workspace",
        system_prompt=get_agent("google_workspace").system_prompt,
        tools=get_langchain_tools("google_workspace", session_id=state["session_id"]),
    )


def _code_executor_node(state: AgentState) -> AgentState:
    return run_react_agent(
        state=state, agent_id="code_executor",
        system_prompt=get_agent("code_executor").system_prompt,
        tools=get_langchain_tools("code_executor", session_id=state["session_id"]),
    )


def _notes_node(state: AgentState) -> AgentState:
    return run_react_agent(
        state=state, agent_id="notes",
        system_prompt=get_agent("notes").system_prompt,
        tools=get_langchain_tools("notes", session_id=state["session_id"]),
    )


def _build_single_node_graph(name: str, fn):
    g = StateGraph(AgentState)
    g.add_node(name, fn)
    g.set_entry_point(name)
    g.add_edge(name, END)
    return g.compile()


def build_general_graph():
    return _build_single_node_graph("general", _general_node)


def build_google_workspace_graph():
    return _build_single_node_graph("google_workspace", _google_workspace_node)


def build_code_executor_graph():
    return _build_single_node_graph("code_executor", _code_executor_node)


def build_notes_graph():
    return _build_single_node_graph("notes", _notes_node)


print("Subgraph factories defined.")

Subgraph factories defined.


## 9. Orchestrator + run_turn

Slimmed port of `surajclaw/agents/orchestrator.py` and `surajclaw/agents/graph.py:run_turn`. No DB persistence (`_persist` skipped). `run_turn` accepts `on_event=print_event` so the cell output shows the streamed token + tool-call frames the WebSocket would emit.

In [10]:
def invoke_agent(invocation: AgentInvocation) -> AgentResult:
    definition = get_agent(invocation.agent_id)
    graph = definition.graph_factory()
    state: AgentState = {
        "session_id": invocation.session_id,
        "source": invocation.source,
        "active_agent": definition.id,
        "user_message": invocation.task,
        "context": dict(invocation.context or {}),
        "account_label": invocation.account_label,
        "step_count": 0,
        "max_steps": definition.max_steps,
        "tool_calls": [], "tool_results": [],
        "agent_results": (invocation.context or {}).get("agent_results", []),
        "agent_trace": [], "agent_messages": [],
        "done": False,
    }
    try:
        final_state = graph.invoke(state)
    except Exception as exc:
        return AgentResult(
            agent_id=definition.id, status="failed",
            output=f"agent graph failed: {exc}",
            structured={"error": type(exc).__name__},
        )
    result = final_state.get("agent_result") or {}
    return AgentResult(
        agent_id=definition.id,
        status=result.get("status", "ok"),
        output=result.get("output", final_state.get("final_response", "")),
        structured=result.get("structured", {}),
    )


def build_orchestrator_graph():
    g = StateGraph(AgentState)

    def invoke_agent_node(state: AgentState) -> AgentState:
        agent_id = state.get("requested_agent") or "general"
        state["active_agent"] = agent_id
        state["step_count"] = state.get("step_count", 0) + 1
        result = invoke_agent(AgentInvocation(
            session_id=state["session_id"],
            source=state.get("source", "web"),
            agent_id=agent_id,
            task=state.get("user_message", ""),
            account_label=(state.get("context") or {}).get("account_label") or state.get("account_label"),
            context={**(state.get("context") or {}), "agent_results": state.get("agent_results", [])},
        ))
        row = result.__dict__
        state.setdefault("agent_results", []).append(row)
        state.setdefault("agent_trace", []).append(
            {"step": state["step_count"], "agent": agent_id, "status": result.status}
        )
        state["last_agent_result"] = row
        state["final_response"] = result.output
        state["agent_result"] = row
        state["model_used"] = result.structured.get("model_used")
        return state

    g.add_node("invoke_agent", invoke_agent_node)
    g.set_entry_point("invoke_agent")
    g.add_edge("invoke_agent", END)
    return g.compile()


def run_turn(session_id: str, message: str, source: str = "web", directives: dict | None = None,
             on_event=None) -> dict:
    """Notebook-friendly run_turn: returns {final, trace, events}."""
    events: list[dict] = []

    def dispatch(payload: dict) -> None:
        events.append(payload)
        if on_event is not None:
            on_event(payload)

    initial: AgentState = {
        "session_id": session_id,
        "source": source,
        "user_message": message,
        "requested_agent": (directives or {}).get("agent"),
        "tool_calls": [], "tool_results": [],
        "agent_results": [], "agent_trace": [],
        "context": {"directives": directives or {}, "on_event": dispatch,
                    "account_label": GOOGLE_ACCOUNT_LABEL},
        "messages": [], "agent_messages": [],
        "step_count": 0, "done": False,
    }
    graph = build_orchestrator_graph()
    state = graph.invoke(initial)
    final = state.get("final_response", "")
    dispatch({"type": "final", "content": final})
    return {"final": final, "trace": state.get("agent_trace", []), "events": events}


def print_event(payload: dict) -> None:
    t = payload.get("type")
    if t == "token":
        print(payload.get("content", ""), end="", flush=True)
    elif t == "tool_call":
        print(f"\n[tool_call] {payload.get('name')} args={payload.get('args')}")
    elif t == "tool_result":
        snippet = (payload.get("content") or "")[:200]
        print(f"[tool_result] {payload.get('name')}: {snippet}")
    elif t == "final":
        print(f"\n[final] {payload.get('content', '')[:200]}")


reset_agents()
print("Orchestrator + run_turn ready. Agents:", [a.id for a in list_agents()])

Orchestrator + run_turn ready. Agents: ['code_executor', 'general', 'google_workspace', 'notes']


## 10. Repro the bug

Run the exact prompt that hit `memory_search` in production. With the V1 (current production) prompts loaded above, expectation is:

* `tool_call` for `memory_search` (or no Gmail call at all)
* No `invoke_google_workspace -> google_gmail_search_messages` chain

That's the bug captured in cells. Helper `tool_call_chain(events)` extracts the routing summary so we can compare runs cleanly.

In [12]:
def tool_call_chain(events: list[dict]) -> list[str]:
    return [f"{e['name']}({e.get('args')})" for e in events if e.get("type") == "tool_call"]


reset_agents()
result_v1 = run_turn("lab-repro", "list are my emails", on_event=print_event)

print("\n\n=== TOOL CALL CHAIN (V1 prompts) ===")
for call in tool_call_chain(result_v1["events"]):
    print(" -", call)
print("\nfinal:", result_v1["final"][:300])

/var/folders/1h/m2mntztj6nj94pk3wmc3bcp00000gn/T/ipykernel_47610/140291694.py:130: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=prompt)



[tool_call] invoke_google_workspace args={'task': 'list my emails'}

[tool_call] google_accounts_list args={}
[tool_result] google_accounts_list: accounts: suraj2308

[tool_call] google_gmail_list_threads args={'limit': 10, 'account_label': 'suraj2308'}
[tool_result] google_gmail_list_threads: Found 10 Gmail threads.

[tool_call] google_gmail_search_messages args={'account_label': 'suraj2308', 'limit': 10, 'query': 'in:inbox'}
[tool_result] google_gmail_search_messages: Found 10 Gmail messages.

[tool_call] google_gmail_get_message args={'message_id': '193634045f271923', 'account_label': 'suraj2308'}
[tool_result] google_gmail_get_message: ERROR (HttpError): tool failed: <HttpError 404 when requesting https://gmail.googleapis.com/gmail/v1/users/me/messages/193634045f271923?format=full&alt=json returned "Requested entity was not found.".

[tool_call] google_gmail_search_messages args={'limit': 5, 'account_label': 'suraj2308'}
[tool_result] google_gmail_search_messages: Found 5 Gmail me

## 11. Fix iteration

Rewrite the General Agent system prompt with explicit routing rules and rewrite the three delegation tool docstrings with exhaustive surface lists. The two together are what the LLM uses to decide which tool to fire — leaving either one thin lets `memory.search` win on personal-data prompts.

After this cell, re-run the repro (cell 22). Expectation: the chain shows `invoke_google_workspace` first, then inside that subagent `google_gmail_search_messages`.

These prompt + docstring strings are the **canonical text** that gets ported into `surajclaw/agents/registry.py` and `surajclaw/agents/subgraphs/general.py` in Phase 2.

In [13]:
GENERAL_PROMPT_V2 = """You are SurajClaw's General Agent, the supervisor of a single-user personal AI.

Routing rules (apply in order, pick the FIRST that matches):

1. If the user asks about their email / inbox / Gmail messages, calendar / events / meetings, Google Drive files / Docs / Sheets, Google Tasks, or Google Contacts, you MUST call `invoke_google_workspace` with a clear task description. Never try to answer these from memory or general knowledge; the user's personal data only lives behind that subagent.
2. If the user asks to run code, execute a shell command, debug a script, or run tests, call `invoke_code_executor`.
3. If the user asks to write, save, search, or list a Markdown note, call `invoke_notes_agent`.
4. Use `web.search` only for public-internet questions (news, definitions, documentation lookup).
5. Use `memory.search` ONLY when the user is asking you to recall something they have explicitly told you to remember in a previous turn (phrases like "do you remember", "what did I save about", "what did we note about"). Do NOT use `memory.search` as a fallback for personal data like emails, calendar, files, or contacts -- those live in the Google Workspace subagent, not in memory.
6. Use `workspace.read_file` / `workspace.write_file` / `workspace.list_files` for files under the local WORKSPACE_DIR only.
7. Use `sandbox.*` tools for direct sandbox access when delegating to `invoke_code_executor` would be overkill (one-shot read).
8. If none of the above match and a tool would not materially help, answer directly.

Other rules:

- Do not claim an external action succeeded unless the corresponding tool or subagent confirmed it in its result.
- When you delegate, pass the user's intent verbatim or near-verbatim as the `task` argument; do not pre-summarize away details the specialist needs.
- If a tool returns an error, acknowledge it, do not pretend it worked, and choose the next safest step.
"""

GOOGLE_PROMPT_V2 = """You are SurajClaw's Google Workspace specialist.

You are invoked by the General Agent for any task touching Gmail, Calendar, Tasks,
Drive, Docs, Sheets, or Contacts on the user's connected Google account.

- Always pass the supplied `account_label` to every tool that takes one. If the
  caller did not supply one, default to "default".
- Prefer read/search/list tools before write/update/delete.
- Gmail tools are read-only in this system; never claim you sent or modified mail.
- For ambiguous requests, call the most general listing tool first
  (e.g. `gmail_search_messages` with an empty query and a small limit) and
  then summarize what you found.
- Summarize structured results in plain English; do not dump raw API JSON at the user.
"""

CODE_PROMPT_V2 = """You are SurajClaw's Code Executor specialist.

You are invoked by the General Agent for any task that needs to run code, a
shell command, or a test.

- Use only the `sandbox.*` tools. Never assume you have the host shell.
- Extract the exact command or code from the user's natural-language request
  before invoking a tool; do not paraphrase commands.
- Always summarize stdout, stderr, exit code, and the next debugging step the
  user should take.
- If a command fails, stop after one diagnostic re-run; do not loop.
"""

NOTES_PROMPT_V2 = """You are SurajClaw's Notes Agent.

You are invoked for any task that asks you to write, update, search, or list
Markdown notes, or to research-then-note.

- Search existing notes first to avoid duplicates.
- For research, use `web.search`, synthesize the useful facts, then write a
  concise Markdown note with sources cited inline.
- Keep notes short, source-aware, and easy to retrieve later (descriptive
  titles, no fluff).
"""

DELEGATION_DOCS_V2 = {
    "google_workspace": (
        "Delegate to the Google Workspace specialist subagent. Use this for ANY task that touches "
        "the user's personal Google account: reading or searching Gmail (inbox, threads, messages, emails); "
        "listing or creating Calendar events / meetings; listing, creating, or updating Google Tasks; "
        "searching Google Drive files; creating or editing Google Docs; creating or updating Google Sheets; "
        "searching Google Contacts. Pass the user's request as `task` and the account label (defaults to the "
        "connected account) as `account_label`. ALWAYS prefer this over memory.search for any question about "
        "the user's email, calendar, files, or contacts."
    ),
    "code_executor": (
        "Delegate to the sandboxed Code Executor subagent. Use this for ANY request that runs code: "
        "shell commands, Python scripts, debugging a snippet, or running a test suite. Pass the user's "
        "raw command or code intent as `task`."
    ),
    "notes": (
        "Delegate to the Notes Agent subagent. Use this when the user asks to write, save, update, search, "
        "or list a Markdown note, or to research a topic and capture the result as a note. Pass the user's "
        "request as `task`."
    ),
}

CURRENT_DELEGATION_DOCS.clear()
CURRENT_DELEGATION_DOCS.update(DELEGATION_DOCS_V2)

reset_agents(
    general_prompt=GENERAL_PROMPT_V2,
    google_prompt=GOOGLE_PROMPT_V2,
    code_prompt=CODE_PROMPT_V2,
    notes_prompt=NOTES_PROMPT_V2,
)

result_v2 = run_turn("lab-fix", "what are my emails", on_event=print_event)
print("\n\n=== TOOL CALL CHAIN (V2 prompts) ===")
for call in tool_call_chain(result_v2["events"]):
    print(" -", call)
print("\nfinal:", result_v2["final"][:400])

/var/folders/1h/m2mntztj6nj94pk3wmc3bcp00000gn/T/ipykernel_47610/140291694.py:130: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=prompt)



[tool_call] invoke_google_workspace args={'task': 'what are my emails'}

[tool_call] google_gmail_list_threads args={'limit': 5, 'account_label': 'default'}
[tool_result] google_gmail_list_threads: ERROR (LookupError): tool failed: no token for account 'default' at /Users/sudarshanrajagopalan/Developer/SurajClaw/surajclaw/google_tokens/default.json. Run: python manage.py google_oauth_login --acc
It seems I'm unable to access your emails because the Google account "default" is not currently authenticated.

To fix this, please run the following command in your terminal:
`python manage.py google_oauth_login --account default`

Once you have completed the authentication process, let me know, and I'll be happy to check your emails for you!{'type': 'text', 'text': '', 'extras': {'signature': 'EjQKMgEMOdbH9EE/4MzqHJbpvv41MqBOaoWdPwYuKTd/pRho/mO4VycyrKM989BYdPv509tn'}, 'index': 0}[tool_result] invoke_google_workspace: google_workspace [ok]: It seems I'm unable to access your emails because th

## 12. Routing test matrix

Run a small set of prompts and assert the tool the General Agent fires first matches the expected target. Keep the runs sequential so the print output reads top-to-bottom for visual diff. If any row prints `MISMATCH`, iterate on cell 24 until everything goes green.

In [14]:
ROUTING_CASES = [
    ("what are my emails", "invoke_google_workspace"),
    ("show me unread mail from this week", "invoke_google_workspace"),
    ("create a calendar event tomorrow at 4pm called sync", "invoke_google_workspace"),
    ("find a file called Q3 plan in my drive", "invoke_google_workspace"),
    ("run python -c 'print(2+2)' for me", "invoke_code_executor"),
    ("write a note titled meeting prep with the agenda below", "invoke_notes_agent"),
    ("do you remember what I said about project status", "memory_search"),
    ("what is the latest stable version of FastAPI", "web_search"),
]


def first_tool(events: list[dict]) -> str:
    for e in events:
        if e.get("type") == "tool_call":
            return e.get("name", "")
    return "<no tool call>"


print(f"{'expected':35} {'observed':35} prompt")
print("-" * 110)

for prompt, expected in ROUTING_CASES:
    out = run_turn(f"lab-matrix-{hash(prompt) & 0xffff}", prompt)
    observed = first_tool(out["events"])
    status = "OK     " if observed == expected else "MISMATCH"
    print(f"{status} {expected:35} {observed:35} {prompt[:55]}")

expected                            observed                            prompt
--------------------------------------------------------------------------------------------------------------


/var/folders/1h/m2mntztj6nj94pk3wmc3bcp00000gn/T/ipykernel_47610/140291694.py:130: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=prompt)


OK      invoke_google_workspace             invoke_google_workspace             what are my emails


/var/folders/1h/m2mntztj6nj94pk3wmc3bcp00000gn/T/ipykernel_47610/140291694.py:130: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=prompt)


OK      invoke_google_workspace             invoke_google_workspace             show me unread mail from this week


...

OK      invoke_notes_agent                  invoke_notes_agent                  write a note titled meeting prep with the agenda below


/var/folders/1h/m2mntztj6nj94pk3wmc3bcp00000gn/T/ipykernel_47610/140291694.py:130: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=prompt)


OK      memory_search                       memory_search                       do you remember what I said about project status


/var/folders/1h/m2mntztj6nj94pk3wmc3bcp00000gn/T/ipykernel_47610/140291694.py:130: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=prompt)


OK      web_search                          web_search                          what is the latest stable version of FastAPI


---

# Architecture v2: explicit StateGraph in WaqtAgent style

The cells above (1-26) use `langgraph.prebuilt.create_react_agent` and a `_subagent_tools` factory that exposes `invoke_google_workspace` etc. as `StructuredTool`s the LLM picks. The cells below tear that out:

- **Each agent** is a hand-written `StateGraph(TurnState)` with two nodes: `agent_llm` (calls `llm.bind_tools(tools).invoke(messages)`) and `tool_executor` (a `ToolNode(tools)`). A conditional edge inspects the last AIMessage for `tool_calls` and routes to either the executor (loop back) or END.
- **Top-level orchestrator** is a `StateGraph(TurnState)` with `GENERAL`, `GOOGLE_WORKSPACE`, `CODE_EXECUTOR`, `NOTES` nodes; each node `.invoke()`s its subgraph. After GENERAL runs, `route_after_general` parses the last assistant message: if it ends with `ROUTE: <TARGET>` the graph hands off to that specialist subgraph; otherwise GENERAL's answer goes to the user.
- **No fake `invoke_*` tools.** Routing is graph control flow, not tool selection.

Pattern matches `WaqtAgent.py:Waqt.__init__` (DEBRIEFER + DEBRIEFER_TOOL_EXECUTOR + SQL_EXECUTOR + DATA_INTERPRETER + PLOT with `add_conditional_edges` and pure-python `route_after_*` functions).

## 14. TurnState

Single TypedDict shared by the orchestrator and every subgraph. `messages` is annotated with `add_messages` so subgraph loops accumulate history correctly. `tool_call_log` is a flat audit list the orchestrator drains for UI frames.

In [ ]:
import operator
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


class TurnState(TypedDict, total=False):
    user_message: str
    messages: Annotated[list[BaseMessage], add_messages]
    tool_call_log: Annotated[list[dict[str, Any]], operator.add]
    route: str | None
    final_response: str
    active_agent: str
    account_label: str | None
    context: dict[str, Any]
    session_id: str
    source: str
    loop_count: int
    max_loops: int


print("TurnState defined.")

## 15. `build_agent_subgraph` factory

Given an `agent_id`, a `system_prompt`, and a list of LangChain tools, returns a compiled `StateGraph(TurnState)` with the explicit shape:

```
agent_llm -> route_after_llm -> {tool_executor -> agent_llm} | END
```

`agent_llm_node` builds its message list fresh from the system prompt + accumulated `state["messages"]`, calls `llm.bind_tools(tools).invoke(...)`, and writes the AIMessage back via `add_messages`. Tool-call frames are appended to `state["tool_call_log"]` so the orchestrator can mirror them to the WebSocket. `tool_executor_node` wraps `ToolNode(tools)` and also appends `tool_result` frames.

A `loop_count` guard caps runaway loops at `max_loops` (default 8) regardless of what the model wants to do.

In [ ]:
def _emit(state: TurnState, payload: dict[str, Any]) -> None:
    """Best-effort callback to ``state['context']['on_event']`` if present."""
    on_event = (state.get("context") or {}).get("on_event")
    if on_event:
        try:
            on_event(payload)
        except Exception:
            pass


def build_agent_subgraph(agent_id: str, system_prompt: str, tools: list[Any], max_loops: int = 8):
    """Compile a per-agent StateGraph: agent_llm + tool_executor + conditional loop."""
    tool_node = ToolNode(tools)

    def agent_llm_node(state: TurnState) -> dict[str, Any]:
        prior = list(state.get("messages") or [])
        if not any(isinstance(m, HumanMessage) for m in prior):
            prior = [HumanMessage(content=state.get("user_message", ""))] + prior
        composed: list[BaseMessage] = [SystemMessage(content=system_prompt), *prior]
        llm = build_llm()
        bound = llm.bind_tools(tools) if tools else llm
        ai_msg: AIMessage = bound.invoke(composed)

        new_log: list[dict[str, Any]] = []
        for call in getattr(ai_msg, "tool_calls", None) or []:
            frame = {
                "type": "tool_call",
                "agent": agent_id,
                "name": call.get("name", ""),
                "args": call.get("args") or {},
                "id": call.get("id") or "",
            }
            new_log.append(frame)
            _emit(state, frame)

        return {
            "messages": [ai_msg],
            "tool_call_log": new_log,
            "loop_count": state.get("loop_count", 0) + 1,
        }

    def tool_executor_node(state: TurnState) -> dict[str, Any]:
        result = tool_node.invoke({"messages": state["messages"]})
        produced = result.get("messages", []) if isinstance(result, dict) else []
        new_log: list[dict[str, Any]] = []
        for tm in produced:
            if isinstance(tm, ToolMessage):
                frame = {
                    "type": "tool_result",
                    "agent": agent_id,
                    "name": getattr(tm, "name", "") or "",
                    "id": getattr(tm, "tool_call_id", "") or "",
                    "content": str(getattr(tm, "content", ""))[:1200],
                }
                new_log.append(frame)
                _emit(state, frame)
        return {"messages": produced, "tool_call_log": new_log}

    def route_after_llm(state: TurnState) -> str:
        if state.get("loop_count", 0) >= max_loops:
            return "FINISH"
        msgs = state.get("messages") or []
        if not msgs:
            return "FINISH"
        last = msgs[-1]
        if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
            return "TOOL"
        return "FINISH"

    g = StateGraph(TurnState)
    g.add_node("agent_llm", agent_llm_node)
    g.add_node("tool_executor", tool_executor_node)
    g.set_entry_point("agent_llm")
    g.add_conditional_edges("agent_llm", route_after_llm, {"TOOL": "tool_executor", "FINISH": END})
    g.add_edge("tool_executor", "agent_llm")
    return g.compile()


print("build_agent_subgraph defined.")

## 16. General agent prompt (with ROUTE protocol)

The General agent keeps its direct tools (`web.search`, `memory.search`, `workspace.*`) and answers most prompts itself. To delegate, it ends its assistant message with a single line of the form `ROUTE: GOOGLE_WORKSPACE` (or `CODE_EXECUTOR` / `NOTES`). The orchestrator parses that tag and hands off.

In [ ]:
GENERAL_PROMPT_V3 = """You are SurajClaw's General Agent, the supervisor of a single-user personal AI.

You have direct access to these tools and should use them when they help:

- `web_search`: public-internet questions (news, definitions, documentation lookup).
- `memory_search`: ONLY when the user asks you to recall something they explicitly told you to remember in a previous turn (e.g. "do you remember", "what did I save about", "what did we note about"). NEVER use memory_search for personal data like emails, calendar, files, or contacts.
- `workspace_read_file` / `workspace_write_file` / `workspace_list_files`: files under the local WORKSPACE_DIR.
- `sandbox_*`: one-shot sandbox reads/writes only.

For anything that is NOT one of the above, you must DELEGATE to a specialist subagent by ending your reply with one of these tags on its own line and nothing after it:

  ROUTE: GOOGLE_WORKSPACE   - for Gmail / inbox / email, Calendar / events / meetings, Google Drive / Docs / Sheets, Google Tasks, Google Contacts.
  ROUTE: CODE_EXECUTOR      - for running code, shell commands, debugging scripts, running tests.
  ROUTE: NOTES              - for writing, updating, searching, or listing Markdown notes; or research-then-note.

When you delegate, restate the user's intent clearly in plain text BEFORE the ROUTE: line so the specialist subagent has full context. Do not call a tool in the same turn you emit a ROUTE tag.

If the question is small-talk, a definition you know, or a clarifying question, just answer directly without any tool or ROUTE tag.

Do not claim an external action succeeded unless a tool you actually called returned a confirming result.
"""

print(f"GENERAL_PROMPT_V3 length={len(GENERAL_PROMPT_V3)}")

## 17. Top-level orchestrator graph (v2)

`build_orchestrator_graph_v2()` wires:

- entry -> `GENERAL` (compiled subgraph)
- `GENERAL` -> conditional edge `route_after_general`:
  - `ANSWER` -> END (general handled it)
  - `ROUTE_GOOGLE_WORKSPACE` -> `GOOGLE_WORKSPACE` subgraph
  - `ROUTE_CODE_EXECUTOR` -> `CODE_EXECUTOR` subgraph
  - `ROUTE_NOTES` -> `NOTES` subgraph
- each specialist -> END

Each top-level node `.invoke()`s a subgraph compiled with `build_agent_subgraph` from cell 32. Subgraphs receive a fresh `messages` slice keyed off `user_message` (or the restated intent the General Agent left in its message right before the `ROUTE:` line).

In [ ]:
import re as _re

ROUTE_PATTERN = _re.compile(r"(?mi)^\s*ROUTE:\s*([A-Z_]+)\s*$")

ROUTE_TO_NODE = {
    "GOOGLE_WORKSPACE": "GOOGLE_WORKSPACE",
    "CODE_EXECUTOR": "CODE_EXECUTOR",
    "NOTES": "NOTES",
}


def _last_ai_text(messages: list[BaseMessage]) -> str:
    for m in reversed(messages or []):
        if isinstance(m, AIMessage):
            content = m.content
            if isinstance(content, list):
                content = "".join(part.get("text", "") if isinstance(part, dict) else str(part) for part in content)
            return str(content or "")
    return ""


def _strip_route_tag(text: str) -> str:
    return ROUTE_PATTERN.sub("", text).rstrip()


def route_after_general(state: TurnState) -> str:
    text = _last_ai_text(state.get("messages") or [])
    m = ROUTE_PATTERN.search(text)
    if not m:
        return "ANSWER"
    target = m.group(1).strip().upper()
    if target not in ROUTE_TO_NODE:
        return "ANSWER"
    return f"ROUTE_{target}"


def _initial_subgraph_state(parent: TurnState, user_message: str) -> dict[str, Any]:
    """Build a fresh state for a specialist subgraph from the parent TurnState."""
    return {
        "user_message": user_message,
        "messages": [],
        "tool_call_log": [],
        "loop_count": 0,
        "max_loops": parent.get("max_loops", 8),
        "active_agent": parent.get("active_agent"),
        "account_label": parent.get("account_label") or GOOGLE_ACCOUNT_LABEL,
        "context": parent.get("context") or {},
        "session_id": parent.get("session_id", ""),
        "source": parent.get("source", "web"),
    }


def _make_specialist_node(agent_id: str, system_prompt: str, allowed_tools: frozenset[str]):
    sub = build_agent_subgraph(
        agent_id,
        system_prompt,
        get_langchain_tools(agent_id),
    )

    def node(state: TurnState) -> dict[str, Any]:
        general_text = _strip_route_tag(_last_ai_text(state.get("messages") or []))
        task = general_text.strip() or state.get("user_message", "")
        sub_state = _initial_subgraph_state(state, task)
        out = sub.invoke(sub_state)
        out_text = _last_ai_text(out.get("messages") or []) or "(no output)"
        return {
            "active_agent": agent_id,
            "final_response": out_text,
            "tool_call_log": out.get("tool_call_log", []),
            "messages": out.get("messages", []),
        }

    return node


def build_orchestrator_graph_v2():
    general_sub = build_agent_subgraph(
        "general",
        GENERAL_PROMPT_V3,
        get_langchain_tools("general"),
    )

    def general_node(state: TurnState) -> dict[str, Any]:
        sub_state = _initial_subgraph_state(state, state.get("user_message", ""))
        out = general_sub.invoke(sub_state)
        text = _last_ai_text(out.get("messages") or [])
        return {
            "active_agent": "general",
            "final_response": _strip_route_tag(text) or text,
            "tool_call_log": out.get("tool_call_log", []),
            "messages": out.get("messages", []),
        }

    google_node = _make_specialist_node("google_workspace", GOOGLE_PROMPT_V2, get_agent("google_workspace").allowed_tools)
    code_node = _make_specialist_node("code_executor", CODE_PROMPT_V2, get_agent("code_executor").allowed_tools)
    notes_node = _make_specialist_node("notes", NOTES_PROMPT_V2, get_agent("notes").allowed_tools)

    g = StateGraph(TurnState)
    g.add_node("GENERAL", general_node)
    g.add_node("GOOGLE_WORKSPACE", google_node)
    g.add_node("CODE_EXECUTOR", code_node)
    g.add_node("NOTES", notes_node)
    g.set_entry_point("GENERAL")
    g.add_conditional_edges(
        "GENERAL",
        route_after_general,
        {
            "ANSWER": END,
            "ROUTE_GOOGLE_WORKSPACE": "GOOGLE_WORKSPACE",
            "ROUTE_CODE_EXECUTOR": "CODE_EXECUTOR",
            "ROUTE_NOTES": "NOTES",
        },
    )
    g.add_edge("GOOGLE_WORKSPACE", END)
    g.add_edge("CODE_EXECUTOR", END)
    g.add_edge("NOTES", END)
    return g.compile()


print("build_orchestrator_graph_v2 defined.")

## 18. `run_turn_v2`

Drives the new orchestrator with `graph.stream(state, stream_mode="updates")` so we get one frame per node finish (no per-token streaming). Each node's `tool_call_log` is mirrored as `tool_call` / `tool_result` frames; we also emit a `node_update` for each node boundary so the UI can show "GENERAL routing -> GOOGLE_WORKSPACE working" live.

In [ ]:
def run_turn_v2(session_id: str, message: str, source: str = "web", on_event=None) -> dict[str, Any]:
    events: list[dict[str, Any]] = []

    def dispatch(payload: dict[str, Any]) -> None:
        events.append(payload)
        if on_event is not None:
            on_event(payload)

    initial: TurnState = {
        "user_message": message,
        "messages": [],
        "tool_call_log": [],
        "loop_count": 0,
        "max_loops": 8,
        "active_agent": "general",
        "account_label": GOOGLE_ACCOUNT_LABEL,
        "context": {"on_event": dispatch},
        "session_id": session_id,
        "source": source,
    }

    graph = build_orchestrator_graph_v2()
    final_state: TurnState = {}
    final_response = ""
    active_agent = "general"
    seen_log = 0

    for update in graph.stream(initial, stream_mode="updates"):
        if not isinstance(update, dict):
            continue
        for node_name, node_state in update.items():
            if not isinstance(node_state, dict):
                continue
            log = node_state.get("tool_call_log") or []
            for frame in log[seen_log:]:
                dispatch(frame)
            if log:
                seen_log = len(log)
            dispatch({"type": "node_update", "node": node_name})
            if node_state.get("final_response"):
                final_response = node_state["final_response"]
            if node_state.get("active_agent"):
                active_agent = node_state["active_agent"]
            final_state = {**final_state, **node_state}

    dispatch({"type": "final", "content": final_response, "active_agent": active_agent})
    return {"final": final_response, "active_agent": active_agent, "events": events, "state": final_state}


def print_event_v2(payload: dict[str, Any]) -> None:
    t = payload.get("type")
    if t == "node_update":
        print(f"\n[node_update] {payload.get('node')}")
    elif t == "tool_call":
        print(f"  [tool_call/{payload.get('agent')}] {payload.get('name')} args={payload.get('args')}")
    elif t == "tool_result":
        snippet = (payload.get("content") or "")[:150]
        print(f"  [tool_result/{payload.get('agent')}] {payload.get('name')}: {snippet}")
    elif t == "final":
        print(f"\n[final/{payload.get('active_agent')}] {payload.get('content', '')[:300]}")


print("run_turn_v2 defined.")

## 19. Routing matrix v2

Re-runs the same prompts as cell 26 against the new graph. Expected behavior:

- emails / calendar / drive: `[node_update GENERAL]` then `ROUTE: GOOGLE_WORKSPACE` -> `[node_update GOOGLE_WORKSPACE]` with at least one Gmail/Calendar/Drive tool call.
- code: GENERAL -> `[node_update CODE_EXECUTOR]` with sandbox_run_python.
- note: GENERAL -> `[node_update NOTES]` with notes_write.
- recall: GENERAL fires `memory_search` directly, no ROUTE tag, ANSWER -> END.
- web: GENERAL fires `web_search` directly, no ROUTE tag.

Helper `route_path(events)` returns the ordered list of node_updates so the assertion is "GENERAL preceded specialist X".

In [ ]:
def route_path(events: list[dict[str, Any]]) -> list[str]:
    return [e["node"] for e in events if e.get("type") == "node_update"]


def specialist_tool_calls(events: list[dict[str, Any]], agent: str) -> list[str]:
    return [e["name"] for e in events if e.get("type") == "tool_call" and e.get("agent") == agent]


ROUTING_CASES_V2 = [
    ("what are my emails", "GOOGLE_WORKSPACE"),
    ("show me unread mail from this week", "GOOGLE_WORKSPACE"),
    ("show me my calendar events for today", "GOOGLE_WORKSPACE"),
    ("find a file called Q3 plan in my drive", "GOOGLE_WORKSPACE"),
    ("run python that prints 2+2", "CODE_EXECUTOR"),
    ("write a note titled meeting prep with two bullets", "NOTES"),
    ("do you remember anything I saved about project status", None),
    ("what is the latest stable version of FastAPI", None),
]

print(f"{'expected':22} {'observed_path':40} prompt")
print("-" * 110)

for prompt, expected_specialist in ROUTING_CASES_V2:
    out = run_turn_v2(f"matrix-v2-{hash(prompt) & 0xffff}", prompt)
    path = route_path(out["events"])
    if expected_specialist is None:
        ok = path == ["GENERAL"]
        expected_label = "GENERAL only"
    else:
        ok = path[:2] == ["GENERAL", expected_specialist]
        expected_label = f"GENERAL -> {expected_specialist}"
    status = "OK    " if ok else "MISS  "
    print(f"{status} {expected_label:22} {' -> '.join(path):40} {prompt[:55]}")